# Gestione di dati IoT

In [ ]:
# Preconfigurazione: connessione
from pymongo import MongoClient
import random
import time
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["iot_system"]
sensor_collection = db["temperature_readings"]

In [ ]:
# Puliamo la collezione per l'esempio
sensor_collection.delete_many({})

In [ ]:
# Funzione per generare dati di temperatura di 3 sensori
def generate_sensor_data():
    sensor_data = [
        {"sensor_id": 1, "temperature": random.uniform(18.0, 30.0), "timestamp": datetime.now()},
        {"sensor_id": 2, "temperature": random.uniform(18.0, 30.0), "timestamp": datetime.now()},
        {"sensor_id": 3, "temperature": random.uniform(18.0, 30.0), "timestamp": datetime.now()},
    ]
    return sensor_data

In [ ]:
# Funzione per inserire i dati e controllare anomalie in tempo reale
def insert_sensor_data(num_readings = 1000, threashold = 29.5):
    for _ in range(num_readings):
        # Genera i dati per i 3 sensori
        sensor_data = generate_sensor_data()

        # Inserimento dati nel db
        sensor_collection.insert_many(sensor_data)

        # Controllo anomali
        for reading in sensor_data:
            if reading["temperature"] > threashold:
                print(f"""Anomalia rilevata - Sensore {reading["sensor_id"]}, con temperatura: {reading["temperature"]:.2f}°C alle {reading["timestamp"]}""")
        
        # Simulazione ritardo inserimento
        time.sleep(0.01)
    
    print(f"Inseriti {num_readings * 3} documenti in sensor_data (3 per lettura)")

# Eseguo inserimento con controllo in tempo reale
insert_sensor_data(1000, threashold=29.5)

In [12]:
# Pulizia collezione per l'esempio
db.drop_collection('temperature_readings')
db.drop_collection('anomalies')

{'nIndexesWas': 1, 'ns': 'iot_system.anomalies', 'ok': 1.0}

In [9]:
# Definizione schema di validazione (max 29.5 °C consentiti in temperature_readings)
validator = {
    "$jsonSchema":{
        "bsonType": "object",
        "required": ["sensor_id", "temperature", "timestamp"],
        "properties":{
            "sensor_id" : {
                "bsonType" : "int"
            },
            "temperature" : {
                "bsonType" : "double",
                "minimum": 0,
                "maximum": 29.5
            },
            "timestamp" : {
                "bsonType" : "date"
            }
        }
    }
}

db.create_collection(
    "temperature_readings",
    validator={"$jsonSchema": validator["$jsonSchema"]}
)

sensor_collection = db["temperature_readings"]
anomalies_collection = db["anomalies"]

## Inserimento con validazione

In [14]:
# Funzione per inserire i dati con validazione
def insert_sensor_data(num_readings = 10, threashold = 29.5):
    for _ in range(num_readings):
        # Genera i dati per i 3 sensori
        sensor_data = generate_sensor_data()

        for reading in sensor_data:
            if reading["temperature"] <= threashold:
                try:
                    sensor_collection.insert_one(reading)
                except Exception as e:
                    print("Errore validazione: ", e)
            else:
                anomalies_collection.insert_one(reading)
                print(f"""Anomalia registrata - Sensore {reading["sensor_id"]}, con temperatura: {reading["temperature"]:.2f}°C alle {reading["timestamp"]}""")
                
        
        # Simulazione ritardo inserimento
        time.sleep(0.01)

# Eseguo inserimento con controllo in tempo reale
insert_sensor_data(10, threashold=29.5)

print(f"Temperature valide inserite: {sensor_collection.count_documents({})}")
print(f"Anomalie inserite: {anomalies_collection.count_documents({})}")

Anomalia registrata - Sensore 1, con temperatura: 29.67°C alle 2026-03-09 22:19:49.873443
Temperature valide inserite: 59
Anomalie inserite: 1
